سلول ۱: آماده‌سازی پروژه

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

project_root = Path("..").resolve()

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

load_dotenv(project_root / ".env")

print("Project root:", project_root)
print("Env loaded")

Project root: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance
Env loaded


سلول ۲: ساخت Pipeline برای Evaluation

In [2]:
import config.settings as settings
from src.pipeline.rag_pipeline import RAGPipeline

QDRANT_LOCAL_PATH = Path(settings.DATA_VECTORSTORE_PATH) / "qdrant_local"
CACHE_FILE = project_root / "data" / "cache" / "rag_cache.json"

rag = RAGPipeline(
    vectorstore_path=QDRANT_LOCAL_PATH,
    collection_name=settings.COLLECTION_NAME,
    cache_file=CACHE_FILE,
    use_cache=True
)

print("RAG pipeline is ready")
print("Vectorstore:", QDRANT_LOCAL_PATH)
print("Collection:", settings.COLLECTION_NAME)
print("Cache file:", CACHE_FILE)

RAG pipeline is ready
Vectorstore: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/vectorstore/qdrant_local
Collection: financial_knowledge_base
Cache file: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/cache/rag_cache.json


سلول ۳: تعریف سؤال‌های ارزیابی

In [3]:
evaluation_questions = [
    {
        "id": "q01",
        "question": "سواد مالی یعنی چه؟",
        "expected_type": "in_scope"
    },
    {
        "id": "q02",
        "question": "آیا سواد مالی فقط سرمایه‌گذاری است؟",
        "expected_type": "in_scope"
    },
    {
        "id": "q03",
        "question": "چرا خانم‌ها به سواد مالی نیاز دارند؟",
        "expected_type": "in_scope"
    },
    {
        "id": "q04",
        "question": "برای مدیریت خرج و دخل چه باید کرد؟",
        "expected_type": "in_scope"
    },
    {
        "id": "q05",
        "question": "رابطه سواد مالی و سینما چیست؟",
        "expected_type": "in_scope"
    },
    {
        "id": "q06",
        "question": "چرا آموزش سواد مالی برای کودکان مهم است؟",
        "expected_type": "in_scope"
    },
    {
        "id": "q07",
        "question": "چرا مدیریت هزینه‌ها در سواد مالی مهم است؟",
        "expected_type": "in_scope"
    },
    {
        "id": "q08",
        "question": "آیا افزایش درآمد همیشه بهتر از کاهش هزینه است؟",
        "expected_type": "in_scope"
    },
    {
        "id": "q09",
        "question": "بهترین روش خرید بیت‌کوین چیست؟",
        "expected_type": "out_of_scope"
    },
    {
        "id": "q10",
        "question": "پیش‌بینی قیمت دلار در ماه آینده چیست؟",
        "expected_type": "out_of_scope"
    }
]

print("Evaluation questions:", len(evaluation_questions))

for item in evaluation_questions:
    print(item["id"], "-", item["expected_type"], "-", item["question"])

Evaluation questions: 10
q01 - in_scope - سواد مالی یعنی چه؟
q02 - in_scope - آیا سواد مالی فقط سرمایه‌گذاری است؟
q03 - in_scope - چرا خانم‌ها به سواد مالی نیاز دارند؟
q04 - in_scope - برای مدیریت خرج و دخل چه باید کرد؟
q05 - in_scope - رابطه سواد مالی و سینما چیست؟
q06 - in_scope - چرا آموزش سواد مالی برای کودکان مهم است؟
q07 - in_scope - چرا مدیریت هزینه‌ها در سواد مالی مهم است؟
q08 - in_scope - آیا افزایش درآمد همیشه بهتر از کاهش هزینه است؟
q09 - out_of_scope - بهترین روش خرید بیت‌کوین چیست؟
q10 - out_of_scope - پیش‌بینی قیمت دلار در ماه آینده چیست؟


سلول ۴: اجرای ارزیابی روی سؤال‌ها

In [4]:
from tqdm import tqdm

evaluation_results = []

for item in tqdm(evaluation_questions):
    result = rag.ask(
        question=item["question"],
        top_k=5
    )

    evaluation_item = {
        "id": item["id"],
        "question": item["question"],
        "expected_type": item["expected_type"],
        "answer": result["answer"],
        "from_cache": result["from_cache"],
        "sources": result["sources"],
        "retrieved_chunks": result["retrieved_chunks"]
    }

    evaluation_results.append(evaluation_item)

print("Evaluation completed")
print("Results:", len(evaluation_results))

100%|██████████| 10/10 [00:19<00:00,  1.96s/it]

Evaluation completed
Results: 10


سلول ۵: نمایش خلاصه نتایج

In [5]:
for item in evaluation_results:
    print("=" * 100)
    print("ID:", item["id"])
    print("Expected:", item["expected_type"])
    print("From cache:", item["from_cache"])
    print("Question:", item["question"])
    print()
    print("Answer:")
    print(item["answer"])
    print()
    print("Sources:")
    for source in item["sources"]:
        print("-", source["title"])

ID: q01
Expected: in_scope
From cache: True
Question: سواد مالی یعنی چه؟

Answer:
سواد مالی، کسب دانش کافی، داشتن نگرش صحیح و اجرای درست آموزه‌های مالی است.  
هدف این دانش، دستیابی به امنیت و رسیدن به آزادی مالی است.  
سواد مالی شامل مفاهیمی مانند خرج و پس‌انداز، اعتبار و بدهی، سرمایه‌گذاری و مدیریت ریسک می‌شود.

Sources:
- فیل سواد مالی
- سیر مطالعاتی سواد مالی؛ گام نخست
- سواد مالی برای خانم‌ها؛ واقعاً ضروری است؟
ID: q02
Expected: in_scope
From cache: True
Question: آیا سواد مالی فقط سرمایه‌گذاری است؟

Answer:
خیر، سواد مالی فقط سرمایه‌گذاری نیست.  
سواد مالی شامل شش سرفصل اصلی است: خرج و پس‌انداز، اعتبار و بدهی، کاریابی و درآمد، سرمایه‌گذاری، مدیریت ریسک و بیمه و تصمیم‌های مالی.  
این مفهوم یک کل درهم تنیده است و فقط به یک بخش محدود مربوط نمی‌شود.

Sources:
- فیل سواد مالی
- سواد مالی برای خانم‌ها؛ ذوب یخ‌ها
ID: q03
Expected: in_scope
From cache: True
Question: چرا خانم‌ها به سواد مالی نیاز دارند؟

Answer:
خانم‌ها به سواد مالی نیاز دارند، چون سواد مالی آن‌ها پایین‌تر از آقایان است و

سلول ۶: ذخیره نتایج ارزیابی

In [6]:
import json
from datetime import datetime

EVALUATION_DIR = project_root / "data" / "evaluation"
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

output_file = EVALUATION_DIR / f"rag_evaluation_{timestamp}.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(evaluation_results, f, ensure_ascii=False, indent=2)

print("Evaluation saved to:")
print(output_file)

Evaluation saved to:
/Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/evaluation/rag_evaluation_20260709_153736.json


سلول ۷: خلاصه آماری ساده

In [7]:
total = len(evaluation_results)

in_scope_count = sum(1 for item in evaluation_results if item["expected_type"] == "in_scope")
out_of_scope_count = sum(1 for item in evaluation_results if item["expected_type"] == "out_of_scope")
from_cache_count = sum(1 for item in evaluation_results if item["from_cache"])

insufficient_message = "در منابع موجود پاسخ کافی پیدا نشد."

correct_out_of_scope = sum(
    1
    for item in evaluation_results
    if item["expected_type"] == "out_of_scope"
    and item["answer"].strip() == insufficient_message
)

print("Total questions:", total)
print("In-scope questions:", in_scope_count)
print("Out-of-scope questions:", out_of_scope_count)
print("From cache:", from_cache_count)
print("Correct out-of-scope rejections:", correct_out_of_scope, "/", out_of_scope_count)

Total questions: 10
In-scope questions: 8
Out-of-scope questions: 2
From cache: 7
Correct out-of-scope rejections: 2 / 2


سلول ۸: بستن Pipeline

In [8]:
rag.close()
print("RAG pipeline closed")

RAG pipeline closed


سلول ۹: اصلاح فایل Pipeline

In [9]:
from pathlib import Path

pipeline_file = project_root / "src" / "pipeline" / "rag_pipeline.py"

pipeline_code = pipeline_file.read_text(encoding="utf-8")

old_code = '''        sources = self.format_sources(retrieved_chunks)

        result = {
            "question": question,
            "answer": answer,
            "sources": sources,
            "retrieved_chunks": retrieved_chunks,
            "top_k": top_k,
            "embedding_model": self.embedding_model,
            "generation_model": self.generation_model,
            "from_cache": False
        }
'''

new_code = '''        insufficient_message = "در منابع موجود پاسخ کافی پیدا نشد."

        if answer.strip() == insufficient_message:
            sources = []
            retrieved_chunks_for_output = []
        else:
            sources = self.format_sources(retrieved_chunks)
            retrieved_chunks_for_output = retrieved_chunks

        result = {
            "question": question,
            "answer": answer,
            "sources": sources,
            "retrieved_chunks": retrieved_chunks_for_output,
            "top_k": top_k,
            "embedding_model": self.embedding_model,
            "generation_model": self.generation_model,
            "from_cache": False
        }
'''

if old_code not in pipeline_code:
    raise ValueError("Target block not found. Check rag_pipeline.py manually.")

pipeline_code = pipeline_code.replace(old_code, new_code)

pipeline_file.write_text(pipeline_code, encoding="utf-8")

print("Updated:", pipeline_file)

Updated: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/src/pipeline/rag_pipeline.py


سلول ۱۰: ری‌استارت یا Reload

In [10]:
import importlib
import src.pipeline.rag_pipeline as rag_pipeline_module

importlib.reload(rag_pipeline_module)

from src.pipeline.rag_pipeline import RAGPipeline

print("RAGPipeline reloaded")

RAGPipeline reloaded


سلول ۱۱: ساخت Pipeline جدید بعد از اصلاح

In [11]:
rag = RAGPipeline(
    vectorstore_path=QDRANT_LOCAL_PATH,
    collection_name=settings.COLLECTION_NAME,
    cache_file=CACHE_FILE,
    use_cache=False
)

print("New RAG pipeline is ready")

New RAG pipeline is ready


سلول ۱۲: تست سؤال خارج از منابع

In [12]:
result = rag.ask("بهترین روش خرید بیت‌کوین چیست؟", top_k=5)

print("Answer:")
print(result["answer"])

print("\nSources count:", len(result["sources"]))
print("Retrieved chunks count:", len(result["retrieved_chunks"]))

Answer:
در منابع موجود پاسخ کافی پیدا نشد.

Sources count: 0
Retrieved chunks count: 0


In [13]:
rag.close()
print("RAG pipeline closed")

RAG pipeline closed
